# Porto TC1 Manual Baseline Notes

This notebook is an evidence artifact for the `tc1_from_scratch` human baseline.
It records dataset inspection and plain-language decisions only.
The final bank-id mapping lives in `manual_selection_worksheet.md`, not in this notebook.


In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv('~/.kaggle/competitions/porto-seguro-safe-driver-prediction/train.csv')
print('shape', train.shape)
print('target rate', train['target'].mean())
print(train.head())


## Sentinel Missingness And Heavy Sparsity

First-pass reasoning:

- Porto uses `-1` as an encoded missing-value sentinel, so that needs to be normalized before thinking about the rest of the preprocessing stack
- a few categorical columns are so sparse that dropping them is easier to justify than trying to rescue them in a first-pass baseline
- keep this structural cleanup without turning the notebook into a mapping guide


In [ ]:
sentinel_cols = [c for c in train.columns if (train[c] == -1).any()]
print('sentinel column count', len(sentinel_cols))
for column in sentinel_cols:
    print(column, round((train[column] == -1).mean(), 4))

train_nan = train.replace(-1, np.nan)
print(train_nan.isnull().mean().sort_values(ascending=False).head(15))


## Calc Features, Remaining Missingness, And Identifier Check

Second-pass reasoning:

- the calc block is a repeated Porto pruning target because it contributes little signal in many strong kernels
- after sparse-column drops, the remaining missing columns split naturally into categorical and numeric groups for simple typed imputation
- stop before the missing-indicator branch and model-specific encoding or interaction choices


In [ ]:
calc_cols = [c for c in train_nan.columns if 'calc' in c]
calc_corr = train_nan[calc_cols].corrwith(train_nan['target']).abs().sort_values(ascending=False)
print('max calc correlation', calc_corr.max())
print(calc_corr.head())

remaining_missing = train_nan.isnull().mean()
print('remaining categorical missing')
print(remaining_missing[[c for c in train_nan.columns if c.endswith('_cat') and remaining_missing[c] > 0]])
print('remaining numeric missing')
print(remaining_missing[[c for c in ['ps_car_11', 'ps_car_12', 'ps_car_14'] if remaining_missing[c] > 0]])
print('id uniqueness', train['id'].nunique(), len(train))


## Manual Takeaways From Inspection

- replace the `-1` sentinel with standard missing values first
- drop the two extreme-missing categorical columns
- drop the weak `ps_calc_*` block
- drop the pure row identifier
- impute the remaining missing categorical and numeric columns with simple type-aware defaults
- stop before explicit missing-indicator creation and model-specific encoding or interaction work in this first-pass baseline
